In [ ]:
import os

# Create main project folder
os.makedirs("titanic-ml-deployment", exist_ok=True)

# Open the project folder
os.chdir("titanic-ml-deployment")

# Create subfolders
os.makedirs("dataset", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

print("Project folders created successfully!")

Project folders created successfully!


In [ ]:
import pandas as pd

# Load Titanic dataset from its actual location
df = pd.read_csv("/content/train.csv")

# Display success message
print("Dataset loaded successfully!")

# Display number of rows and columns
print("Dataset Shape:", df.shape)

# Display first 5 records
df.head()

Dataset loaded successfully!
Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
!pip install -q pandas numpy scikit-learn joblib fastapi uvicorn requests nest-asyncio

In [ ]:
import os
import time
import shutil
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

os.makedirs("dataset", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
import shutil
import os

# Copy the dataset from '/content/' to our project's 'dataset' directory
if os.path.exists("/content/train.csv"):
    shutil.copy("/content/train.csv", "dataset/train.csv")

# Load the dataset
df = pd.read_csv("dataset/train.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
print("Dataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSurvival Distribution:")
print(df["Survived"].value_counts())

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          6

In [ ]:
# Separate features and target

X = df.drop(columns=["Survived"])
y = df["Survived"]

# First split: 70% training, 30% remaining

X_train, X_remaining, y_train, y_remaining = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Second split: 15% testing, 15% unseen

X_test, X_unseen, y_test, y_unseen = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    random_state=42,
    stratify=y_remaining
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Unseen records:", len(X_unseen))

Training records: 623
Testing records: 134
Unseen records: 134


In [ ]:
# Select useful features

numerical_features = [
    "Pclass", "Age", "SibSp", "Parch", "Fare"
]

categorical_features = [
    "Sex", "Embarked"
]

# Numerical preprocessing

numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing

preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

# Complete ML pipeline

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Train model

model.fit(X_train, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [ ]:
# Predict test data

y_pred = model.predict(X_test)

# Calculate accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 83.58 %

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.90      0.87        82
           1       0.83      0.73      0.78        52

    accuracy                           0.84       134
   macro avg       0.83      0.82      0.82       134
weighted avg       0.84      0.84      0.83       134



In [ ]:
joblib.dump(model, "model.pkl")

print("Model saved successfully as model.pkl")

Model saved successfully as model.pkl


In [ ]:
# Load saved model

loaded_model = joblib.load("model.pkl")

# Select one unseen passenger

online_record = X_unseen.iloc[[0]]

# Generate prediction

prediction = loaded_model.predict(online_record)[0]

print("Online Inference Result")
print("-----------------------")

print("Passenger Details:")
display(online_record)

print(
    "Prediction:",
    "Survived" if prediction == 1 else "Did Not Survive"
)

Online Inference Result
-----------------------
Passenger Details:


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
455,456,3,"Jalsevac, Mr. Ivan",male,29.0,0,0,349240,7.8958,NaN,C


Prediction: Did Not Survive


In [ ]:
# Select 20 unseen passengers

batch_records = X_unseen.iloc[:20].copy()

# Generate predictions together

batch_predictions = loaded_model.predict(batch_records)

# Create results dataframe

batch_results = batch_records.copy()

batch_results["Predicted_Survived"] = batch_predictions

batch_results["Prediction_Label"] = [
    "Survived" if p == 1 else "Did Not Survive"
    for p in batch_predictions
]

# Display results

display(batch_results)

# Save CSV file

batch_results.to_csv(
    "batch_predictions.csv",
    index=False
)

print("Batch predictions saved successfully!")

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Predicted_Survived,Prediction_Label
455,456,3,"Jalsevac, Mr. Ivan",male,29.0,0,0,349240,7.8958,NaN,C,0,Did Not Survive
607,608,1,"Daniel, Mr. Robert Williams",male,27.0,0,0,113804,30.5000,NaN,S,1,Survived
142,143,3,"Hakkarainen, Mrs. Pekka Pietari (Elin Matilda ...",female,24.0,1,0,STON/O2. 3101279,15.8500,NaN,S,0,Did Not Survive
883,884,2,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,NaN,S,0,Did Not Survive
835,836,1,"Compton, Miss. Sara Rebecca",female,39.0,1,1,PC 17756,83.1583,E49,C,1,Survived
593,594,3,"Bourke, Miss. Mary",female,NaN,0,2,364848,7.7500,NaN,Q,1,Survived
353,354,3,"Arnold-Franchi, Mr. Josef",male,25.0,1,0,349237,17.8000,NaN,S,0,Did Not Survive
634,635,3,"Skoog, Miss. Mabel",female,9.0,3,2,347088,27.9000,NaN,S,0,Did Not Survive
184,185,3,"Kink-Heilmann, Miss. Luise Gretchen",female,4.0,0,2,315153,22.0250,NaN,S,1,Survived
718,719,3,"McEvoy, Mr. Michael",male,NaN,0,0,36568,15.5000,NaN,Q,0,Did Not Survive


Batch predictions saved successfully!


In [ ]:
# Select 10 unseen passengers

stream_records = X_unseen.iloc[:10]

print("Starting Streaming Inference...")
print("--------------------------------")

for i in range(len(stream_records)):

    # Get one passenger

    passenger = stream_records.iloc[[i]]

    # Predict one passenger

    prediction = loaded_model.predict(passenger)[0]

    result = (
        "Survived"
        if prediction == 1
        else "Did Not Survive"
    )

    print(
        f"Passenger {i+1}: Prediction = {result}"
    )

    # One second delay

    time.sleep(1)

print("--------------------------------")
print("Streaming Inference Completed!")

Starting Streaming Inference...
--------------------------------
Passenger 1: Prediction = Did Not Survive
Passenger 2: Prediction = Survived
Passenger 3: Prediction = Did Not Survive
Passenger 4: Prediction = Did Not Survive
Passenger 5: Prediction = Survived
Passenger 6: Prediction = Survived
Passenger 7: Prediction = Did Not Survive
Passenger 8: Prediction = Did Not Survive
Passenger 9: Prediction = Survived
Passenger 10: Prediction = Did Not Survive
--------------------------------
Streaming Inference Completed!


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal
import nest_asyncio
import uvicorn
import threading

app = FastAPI(
    title="Titanic Survival Prediction API",
    description="Predict passenger survival using Machine Learning",
    version="1.0"
)

# Pydantic input validation

class Passenger(BaseModel):

    Pclass: int = Field(ge=1, le=3)
    Sex: Literal["male", "female"]
    Age: float = Field(gt=0, le=100)
    SibSp: int = Field(ge=0)
    Parch: int = Field(ge=0)
    Fare: float = Field(ge=0)
    Embarked: Literal["C", "Q", "S"]


# Health endpoint

@app.get("/health")
def health():

    return {
        "status": "healthy",
        "message": "Titanic ML API is running"
    }


# Prediction endpoint

@app.post("/predict")
def predict_survival(passenger: Passenger):

    # Convert input to dataframe

    passenger_data = pd.DataFrame([passenger.model_dump()])

    # Predict

    prediction = loaded_model.predict(passenger_data)[0]

    return {
        "prediction": int(prediction),
        "result": (
            "Survived"
            if prediction == 1
            else "Did Not Survive"
        )
    }


print("FastAPI application created successfully!")

FastAPI application created successfully!


In [ ]:
nest_asyncio.apply()

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

time.sleep(3)

print("FastAPI server started!")

INFO:     Started server process [3315]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


FastAPI server started!


In [ ]:
import requests

# Test health endpoint

health_response = requests.get(
    "http://127.0.0.1:8000/health"
)

print("Health API Response:")
print(health_response.json())

INFO:     127.0.0.1:39740 - "GET /health HTTP/1.1" 200 OK
Health API Response:
{'status': 'healthy', 'message': 'Titanic ML API is running'}


In [ ]:
# Valid passenger details

passenger_data = {
    "Pclass": 3,
    "Sex": "male",
    "Age": 25,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=passenger_data
)

print("Prediction API Response:")
print(response.json())

INFO:     127.0.0.1:39966 - "POST /predict HTTP/1.1" 200 OK
Prediction API Response:
{'prediction': 0, 'result': 'Did Not Survive'}


In [ ]:
invalid_data = {
    "Pclass": 5,
    "Sex": "male",
    "Age": 25,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25,
    "Embarked": "S"
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=invalid_data
)

print("Invalid Request Status:", response.status_code)
print("Response:", response.json())

INFO:     127.0.0.1:55992 - "POST /predict HTTP/1.1" 422 Unprocessable Content
Invalid Request Status: 422
Response: {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'Pclass'], 'msg': 'Input should be less than or equal to 3', 'input': 5, 'ctx': {'le': 3}}]}


In [ ]:
from google.colab.output import eval_js

public_url = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("Swagger UI:", public_url + "/docs")

Swagger UI: https://8000-m-s-usc1b1-8s5qauoy14z4-b.us-central1.prod.r.colab.dev/docs
